# Directors

## Imports

In [2]:
import os
import sys
import itertools

import cv2
import duckdb
import requests
import numpy as np
import pandas as pd
import sqlalchemy as db
import plotly.express as px
import plotly.graph_objects as go
import scipy.ndimage as ndimage
import matplotlib.pyplot as plt
from PIL import Image
from scipy.spatial import distance
from huggingface_hub import hf_hub_download
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm 
from sklearn.linear_model import LinearRegression

sys.path.append(os.path.abspath(os.path.join('..')))

from utils import *

/home/amos/anaconda3/envs/face/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# if os.path.exists('utils.py'):
#     os.remove('utils.py')

# raw_url = "https://raw.githubusercontent.com/astaileyyoung/CineFace/research/research/utils.py"

# response = requests.get(raw_url)

# if response.status_code == 200:
#     with open("utils.py", "wb") as f:
#         f.write(response.content)
#     import utils
#     print("✅ Success! utils.py is now actual code.")
# else:
#     print(f"❌ Failed to download. Error code: {response.status_code}")

## Setup

## Load Data

In [4]:
# dw_local_path = hf_hub_download(
#     repo_id="astaileyyoung/CineFaceDB",
#     filename="CineFaceDW.duckdb",
#     repo_type="dataset",
#     local_dir="."
# )

In [5]:
# conn = duckdb.connect("/home/amos/datasets/CineFace/CineFaceDW.duckdb")

In [6]:
username = "amos"
password = "M0$hicat"
host = "192.168.0.131"
port = "3306"
database = "CineFaceDW"
connection_string = f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}'
engine = db.create_engine(connection_string)
conn = engine.connect()

In [7]:
# df = pd.read_sql_query("SELECT * FROM vwWorksByDirector WHERE kind = 'movie'", conn)
# df

In [8]:
# g = df.groupby('person_id')[[
#     'z_size', 
#     'z_size_g',
#     'z_v_size', 
#     'z_v_size_g',
#     'person_name', 
#     'person_id',
#     'pct_mc',
#     'z_f_per_fr',
#     'z_f_per_fr_g',
#     'z_v_f_per_fr',
#     'z_v_f_per_fr_g',
#     'z_gini',
#     "z_gini_g",
#     'z_dist',
#     'z_dist_g',
#     'z_disp',
#     'z_disp_g',
#     'z_v_dist',
#     'z_v_dist_g',
#     'z_vert',
#     'z_vert_g',
#     'z_h_spread',
#     'z_h_spread_g'

#     ]].agg(
#     {
#         "z_size": ["min", "mean", "max"],
#         "z_size_g": "mean",
#         "z_v_size": ["min", "mean", "max"],
#         "z_v_size_g": "mean",
#         "pct_mc": "mean",
#         "z_f_per_fr": "mean",
#         "z_f_per_fr_g": "mean",
#         "z_v_f_per_fr": "mean",
#         "z_v_f_per_fr_g": "mean",
#         "z_gini": "mean",
#         "z_gini_g": "mean",
#         "z_dist": "mean",
#         "z_dist_g": "mean",
#         "z_disp": "mean",
#         "z_disp_g": "mean",
#         "z_v_dist": "mean",
#         "z_v_dist_g": "mean",
#         "z_vert": "mean",
#         "z_vert_g": "mean",
#         "z_h_spread": "mean",
#         "z_h_spread_g": "mean",
#         "person_name": "max",
#         "person_id": "count"
#     }
# ).rename(
#     {
#         "person_id": "cnt"
#     }, axis=1
# )
# g = g[g['cnt']['count'] > 5]
# g.columns = ['_'.join(col).strip() for col in g.columns.values]
# g = g.reset_index()
# g['z_size_range'] = g['z_size_max'] - g['z_size_min']
# g['z_v_size_range'] = g['z_v_size_max'] - g['z_v_size_min']
# g = g.rename({"person_name_max": "name"}, axis=1)
# g

In [9]:
g = pd.read_sql_query("SELECT * FROM vwDirectorStats WHERE cnt > 5;", conn)
g

,person_id,name,cnt,avg_f_per_fr,z_size_mv_mean,z_size_mv_g_mean,z_size_mv_std,z_size_mv_g_std,z_top1_mv_mean,z_top1_mv_g_mean,...,z_v_density_mv_mean,z_v_density_mv_g_mean,z_pct_top1_mv_mean,z_pct_top1_mv_g_mean,z_v_pct_top1_mv_mean,z_v_pct_top1_mv_g_mean,z_pct_face_mv_mean,z_pct_face_mv_g_mean,z_v_pct_face_mv_mean,z_v_pct_face_mv_g_mean


## Analysis

In [10]:
def plot_directors(df, x, y, trendline_options=None, hover_data=None, xaxis=None, yaxis=None, dst=None, width=800, height=600):
    if not trendline_options:
        trendline_options = {}

    fig = px.scatter(
        df, 
        x=x, 
        y=y, 
        color_discrete_sequence=["#79b8b8"], 
        hover_data=hover_data,
        trendline='ols',
        trendline_options=trendline_options
    )
    fig.update_layout(layout)
    fig.update_layout(
        xaxis=dict(title=x if not xaxis else xaxis),
        yaxis=dict(title=y if not yaxis else yaxis)
    )
    # fig.update_traces(line_width=4)
    fig.update_traces(marker=dict(size=8, line=dict(width=1, color="white")))
    fig.update_traces(selector=dict(mode="lines"), line=dict(dash="dash", color="#d81275", width=5), name="Trend")

    if dst:
        fig.write_image(dst, width=width, height=height, scale=2)
    fig.show()


### Overall

#### Face Size vs. Size Variance

The first measure is relativley simple; it's simply the average size of faces within a film averaged over a director's career. This is plotted against how much the size of the face varies within a film. As we can see, the correlation between these two variables is rather tight.

In [6]:
["title", "imdb_id", "directors", "year"]

['title', 'imdb_id', 'directors', 'year']

In [1]:
x = "z_size_mv_g_mean"
y = "z_v_size_mv_g_mean"
plot_directors(g, x, y, xaxis="Avg. Size", yaxis="Std. Size", dst="./data/size_vs_variance.png", hover_data=["name"])

NameError: name 'plot_directors' is not defined

What might be an explanation? Well, the larger the average face size, the more 'dynamic range' the director has available. A director who favors long shots (like Chaplin) is mathematically restricted to a narrow band of variance because the face is already small. A director who favors close-ups (like Bergman) has the 'spatial budget' to swing wildly between different scales. Therefore, shot-size variety is a luxury of the close-up.

#### Face Size vs. Career Volatility

This measures the average scale of the face within a film against a director's tendency to vary that scale *betweeen* films. A high score on the x axis means that the director highly privileges close-ups while a low score means they favor long shots. A high score on the y axis means that a director changes this tendency between films while a low score means that the director remains largely consistent in their shot scales across their career. 

In [11]:
x = "z_size_g_mean"
y = "z_size_g_std"
plot_directors(g, x, y, hover_data=["name"])

In [12]:
g[["name", "z_size_g_mean"]].sort_values('z_size_g_mean')[:10]

,name,z_size_g_mean
312,Georg af Klercker,-1.134680
35,Buster Keaton,-1.096414
172,Kenji Mizoguchi,-0.995902
19,Charlie Chaplin,-0.981539
205,Fred C. Brannon,-0.894744
159,Howard Bretherton,-0.872640
16,Fred Niblo,-0.839127
201,Hal Walker,-0.837701
135,Ray Taylor,-0.827053
10,Victor Sjöström,-0.820912


In [13]:
g[['name', 'z_size_g_mean']].sort_values('z_size_g_mean', ascending=False)[:10]

,name,z_size_g_mean
302,Tony Scott,1.839268
304,Wong Kar-Wai,1.594032
307,Christopher Nolan,1.376913
311,Paul Thomas Anderson,1.109507
291,David Lynch,1.097792
249,Ingmar Bergman,1.012328
309,Denis Villeneuve,0.952283
303,Ridley Scott,0.951768
298,Michael Mann,0.886773
301,Roger Donaldson,0.871443


#### Face Variance vs. Career Volatility

This plot represents how the variance in face size within a film contrasts with how consistently the face size varies *between* movies. Ingmar Bergman, located in the top right corner, not only varies his face sizes within a film, but between his films. 

In [14]:
x = "z_v_size_g_mean"   # How much do face sizes vary within a movie
y = "z_v_size_g_std"    # How much does a director vary his/her shot scale from one film to the next
plot_directors(g, x, y, hover_data=["name"])

#### Face Size Variance vs. Face Variance Variance

In [15]:
"""
On the x axis is the STDDEV of average face size. How much does a director vary their face sizes between films. 
If a director always uses the same shot scale across their career--close-ups, for instance--they will have a low value on the x axis.

On the y axis is the STDDEV of face size variance. How much do the face sizes vary within a film, and how much does this change between films.
If a director varies their scales a lot within a movie, how much do they do this across their career.
"""
x = 'z_size_g_std'
y = "z_v_size_g_std"    
plot_directors(g, x, y, hover_data=["name"])

In [16]:
g[["name", "z_v_size_g_std"]].sort_values(by="z_v_size_g_std")[:10]

,name,z_v_size_g_std
140,Ralph Staub,0.089711
35,Buster Keaton,0.096517
127,Bernard B. Ray,0.103380
205,Fred C. Brannon,0.104234
312,Georg af Klercker,0.108275
145,Philip Ford,0.119732
90,Louis King,0.120789
159,Howard Bretherton,0.126228
57,Howard Higgin,0.126318
38,Albert S. Rogell,0.126537


In [17]:
g[["name", "z_v_size_g_std"]].sort_values(by="z_v_size_g_std", ascending=False)[:10]

,name,z_v_size_g_std
273,Sergio Leone,1.180246
249,Ingmar Bergman,1.106605
291,David Lynch,1.059886
308,Quentin Tarantino,1.000777
187,Emeric Pressburger,0.878536
316,Yorgos Lanthimos,0.865132
304,Wong Kar-Wai,0.851508
225,Vittorio De Sica,0.798209
243,Robert Parrish,0.794337
309,Denis Villeneuve,0.772202


In [18]:
X = g['z_size_g_std'].values.reshape(-1, 1)
y = g['z_v_size_g_std'].values
model = LinearRegression().fit(X, y)
# 2. Calculate the 'Expected' variance for each director
g['expected_v_g_std'] = model.predict(X)
g['style_pivot_g'] = g['z_v_size_g_std'] - g['expected_v_g_std']

X = g['z_size_std'].values.reshape(-1, 1)
y = g['z_v_size_std'].values
model = LinearRegression().fit(X, y)
# 2. Calculate the 'Expected' variance for each director
g['expected_v_std'] = model.predict(X)
g['style_pivot'] = g['z_v_size_g_std'] - g['expected_v_std']

In [19]:
g.sort_values(by='style_pivot', ascending=True)[["name", "style_pivot"]]

,name,style_pivot
77,Norman Z. McLeod,-1.206140
66,Mervyn LeRoy,-1.066242
214,Delmer Daves,-0.953729
241,Terence Young,-0.925004
5,Chester M. Franklin,-0.885382
...,...,...
273,Sergio Leone,0.050290
287,Brian De Palma,0.068245
316,Yorgos Lanthimos,0.089761
225,Vittorio De Sica,0.121084


#### Face Size vs. Avg. Distance

As you can see below, face size and distance to center of the frame are slightly inversely correlated. This makes sense intuitively because the larger a face is, the less room there is to play within the frame. 

In [20]:
x = "z_size_g_mean"
y = "z_dist_g_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Face Size Variance vs. Horizontal Spread Variance

In [21]:
x = "z_v_size_g_mean"
y = "z_h_spread_g_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Faces per Frame vs. Face Size

This result was a little surprising to me. Yes, there is an inverse relationship between the number of faces per frame and the size of the face. However, it's weaker than I expected. My guess would be that this is because we're aggregating at the film level instead of the frame level. I imagine if we aggregated by frame we might see a different result. There are two significant outliers: Stanley Kramer and Ingmar Bergman. Kramer puts a lot of faces into the frame, but maintains an average face size. This means that the frame is packed with faces. On the other end, Bergman is in the bottom 7 of faces per frame, but is an outlier on face size, as we've already seen.

In [22]:
x = "z_f_per_fr_mean"
y = "z_size_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Faces per Frame vs. Face Size Variance

Because average face size and avg face variance are highly correlated, the relationship between variance and faces per frame is essentially the same plot.

In [23]:
x = "z_f_per_fr_mean"
y = "z_v_size_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Faces per Frame vs. Avg Distance from Center

There is a mild positive relationship between faces per frame and the average distance to the center of the frame. 

In [58]:
x = "z_f_per_fr_g_mean"
y = "z_dist_g_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Faces per Frame vs. Variance of Distance to Center

In [59]:
x = "z_f_per_fr_g_mean"
y = "z_v_dist_g_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Faces per Frame vs. Dispersion

In [26]:
x = "z_f_per_fr_g_mean"
y = "z_disp_g_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Faces per Frame vs. Gini

In [27]:
x = "z_f_per_fr_g_mean"
y = "z_gini_g_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Faces per Frame vs. Variance of Faces per Frame

In [28]:
x = "z_f_per_fr_g_mean"
y = "z_v_f_per_fr_g_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Distance vs. Variance Distance

In [60]:
x = "z_dist_g_mean"
y = "z_v_dist_g_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Distance from Center vs. Gini

In [29]:
x = "z_dist_g_mean"
y = "z_gini_g_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Distance from Center vs. Dispersion

In [30]:
x = "z_dist_g_mean"
y = "z_disp_g_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Distance from Center Average vs. Distance from Center Variance

Somewhat surprising result: the average distance of a face from the center of the image is completely unrelated to how 

In [31]:
x = "z_dist_g_mean"
y = "z_v_dist_g_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Distance from Center Film Variance vs. Distance from Center Career Variance

In [32]:
x = "z_dist_g_std"
y = "z_v_dist_g_std"
plot_directors(g, x, y, hover_data=["name"])

#### Distance from Center Variance vs. Gini

In [33]:
x = "z_v_dist_g_mean"
y = "z_gini_g_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Distance from Center Variance vs. Dispersion

In [34]:
x = "z_v_dist_g_mean"
y = "z_disp_g_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Distance from Center Variance vs. Vertical Discipline

In [35]:
x = "z_v_dist_g_mean"
y = "z_vert_g_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Distance to Center vs. Horizontal Spread

In [36]:
x = "z_v_dist_g_mean"
y = "z_h_spread_g_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Vertical Discipline vs. Dispersion

In [37]:
x = "z_vert_g_mean"
y = "z_disp_g_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Vertical Discipline vs. Gini

In [38]:
x = "z_vert_g_mean"
y = "z_gini_g_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Gini vs. Dispersion

In [39]:
x = "z_gini_g_mean"
y = "z_disp_g_mean"
plot_directors(g, x, y, hover_data=["name"])

In [40]:
g[["name", "z_disp_g_mean"]].sort_values("z_disp_g_mean")[:10]

,name,z_disp_g_mean
231,Yasujirō Ozu,-1.309938
307,Christopher Nolan,-1.146228
310,Ethan Coen,-1.069346
35,Buster Keaton,-1.018091
15,Alan Crosland,-0.999168
4,Sidney Franklin,-0.955274
45,E. Mason Hopper,-0.841083
309,Denis Villeneuve,-0.792212
34,Malcolm St. Clair,-0.787870
65,George Archainbaud,-0.787174


In [41]:
g[["name", "z_disp_g_mean"]].sort_values("z_disp_g_mean", ascending=False)[:10]

,name,z_disp_g_mean
268,Stanley Kramer,1.439756
211,Robert Rossen,1.352742
276,Akira Kurosawa,1.214473
293,Steven Spielberg,1.190123
257,Federico Fellini,1.112174
253,Ken Hughes,0.954112
295,Martin Scorsese,0.904560
287,Brian De Palma,0.845302
221,Richard Fleischer,0.823208
263,Sidney Lumet,0.791781


#### Gini vs. Horizontal Spread

In [42]:
x = "z_gini_g_mean"
y = "z_h_spread_mean"
plot_directors(g, x, y, hover_data=["name"])

### By Year

#### Face Size vs. Size Variance

In [43]:
x = "z_size_mean"
y = "z_v_size_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Face Size vs. Avg. Distance

In [44]:
x = "z_size_mean"
y = "z_dist_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Face Size vs. Variance Distance

In [45]:
x = "z_size_mean"
y = "z_dist_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Faces per Frame vs. Face Size

In [46]:
x = "z_f_per_fr_mean"
y = "z_size_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Face Size vs. Gini Score

In [47]:
x = "z_size_mean"
y = "z_gini_mean"
plot_directors(g, x, y, hover_data=["name"])

#### Face Size Variance vs. Gini

In [48]:
x = "z_dist_g_mean"
y = "z_vert_g_mean"
plot_directors(g, x, y, hover_data=["name"])

### Directors

#### Martin Scorsese -- Master of Ensemble Framing

Scorsese presents an interesting case. On a couple metrics, Scorsese is not only not exceptional, but is almost completely average. This is particularly evident when we look at the relationship between the average face size and the average variation in face size, as seen below.

In [52]:
name = "Martin Scorsese"

##### Face Size vs. Face Variance

On both size and variance, Scorsese is almost completely average. Scorsese also remains relatively consistent across his films, suggesting that 

In [56]:
x = "z_size_std"
y = "z_v_size_std"
plot_directors_against_sample(g, x, y, [name])

In [54]:
g['size_percentile'] = g['z_size_std'].rank(pct=True) * 100
temp = g[g['name'] == "Martin Scorsese"]
temp['size_percentile'].values[0]

38.801261829652994

##### Average Distance from Center vs. Variance Distance from Center

In [57]:
x = "z_dist_mean"
y = "z_v_dist_mean"
plot_directors_against_sample(g, x, y, [name])

In [ ]:
temp = df[df['person_name'] == "Martin Scorsese"]
px.bar(temp, x='year', y='z_gini')

Only one of Scorsese's movies has a positive Z-score. 

In [ ]:
x = "year"
y = "z_gini"
temp = df[df['person_name'] == "Martin Scorsese"]
fig = px.scatter(temp, 
                 x=x,
                 y=y,
                 hover_name="person_name",
                 hover_data="title",
                 color_discrete_sequence=["#79b8b8"],
                 trendline="ols")
fig.update_layout(layout)
fig.update_layout(
    xaxis=dict(title=x), 
    yaxis=dict(title=y), 
    title=dict(
        text=f"{x} vs. {y}",
        y=0.95
    )
)
fig.update_traces(marker=dict(size=8, line=dict(width=1, color="white")))
fig.update_traces(selector=dict(mode="lines"), line=dict(dash="dash", color="#d81275", width=5), name="Trend")
fig.show()

In [ ]:
x = "z_f_per_fr_mean"
y = "z_gini_mean"
plot_director_against_sample(g, x, y, name)
# fig = px.scatter(g, 
#                  x=x,
#                  y=y,
#                  hover_name="name",
#                  color_discrete_sequence=["#79b8b8"],
#                  trendline="ols")
# fig.update_layout(layout)
# fig.update_traces(marker=dict(size=8, line=dict(width=1, color="white")))
# fig.update_traces(selector=dict(mode="lines"), line=dict(dash="dash", color="#d81275", width=5), name="Trend")
# temp = g[g['name'] == "Martin Scorsese"]
# fig.add_trace(go.Scatter(x=temp[x], 
#                          y=temp[y], 
#                          marker=dict(
#                              color="red", 
#                              size=16, 
#                              line=dict(
#                                  color="white",
#                                  width=2
#                              )
#                              )))
# fig.show()

As we can see, Scorsese is almost perfectly average with regard to both average size and average variance. However, if we examine framing, the position of faces within the image, a different picture emerges. When we look at the average Gini score across Scorsese's filmography, we see that Scorsese is, in fact, an outlier.

In [ ]:
x = "z_avg_face_size_mean"
y = "z_composition_gini_mean"
fig = px.scatter(g, 
                 x=x,
                 y=y,
                 hover_name="person_name_max",
                 color_discrete_sequence=["#79b8b8"],
                 trendline="ols")
fig.update_layout(layout)
fig.update_layout(
    xaxis=dict(title=x), 
    yaxis=dict(title=y), 
    title=dict(
        text=f"{x} vs. {y}",
        y=0.95
    )
)
fig.update_traces(marker=dict(size=8, line=dict(width=1, color="white")))
fig.update_traces(selector=dict(mode="lines"), line=dict(dash="dash", color="#d81275", width=5), name="Trend")
temp = g[g['person_name_max'] == "Martin Scorsese"]
fig.add_trace(go.Scatter(x=temp[x], 
                         y=temp[y], 
                         marker=dict(
                             color="red", 
                             size=16, 
                             line=dict(
                                 color="white",
                                 width=2
                             )
                             )))
fig.show()

ValueError: Value of 'x' is not the name of a column in 'data_frame'. Expected one of ['person_id', 'name', 'cnt', 'z_size_mean', 'z_size_g_mean', 'z_size_std', 'z_size_g_std', 'z_v_size_mean', 'z_v_size_g_mean', 'z_v_size_std', 'z_v_size_g_std', 'pct_mc_mean', 'pct_mc_std', 'z_f_per_fr_mean', 'z_f_per_fr_g_mean', 'z_f_per_fr_g_std', 'z_v_f_per_fr_mean', 'z_v_f_per_fr_g_mean', 'z_v_f_per_fr_g_std', 'z_gini_mean', 'z_gini_g_mean', 'z_gini_g_std', 'z_dist_mean', 'z_dist_g_mean', 'z_dist_g_std', 'z_disp_mean', 'z_disp_g_mean', 'z_disp_g_std', 'z_v_dist_mean', 'z_v_dist_g_mean', 'z_v_dist_g_std', 'z_vert_mean', 'z_vert_g_mean', 'z_vert_g_std', 'expected_v_g_std', 'style_pivot_g', 'expected_v_std', 'style_pivot', 'size_percentile'] but received: z_avg_face_size_mean

In [ ]:
g['gini_percentile'] = g['z_composition_gini_mean'].rank(pct=True) * 100
temp = g[g['person_name_max'] == "Martin Scorsese"]
temp['gini_percentile'].values[0]

0.3968253968253968

##### Gridmap

Scorsese is not simply an outlier on gini score--he's at the very bottom. This is an interesting finding because Scorsese is quite average regarding the size and variance of the faces yet an outlier on gini score. Why is this the case? And what does this say about Scorsese's framing? Let's look at a gridmap of Scorsese's framing.

In [ ]:
create_gridmap_from_director("Martin Scorsese", dw_conn, layout=layout)

/tmp/ipykernel_6961/3450768217.py:7: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



Scorsese places the faces predominantly in the middle-top and middle-center. This is consistent with intuition--most faces are framed in the middle of the frame. So why does Scorsese score so low on gini score? Let's take a look at the gridmap of the historical average. 

In [ ]:
plot_sample_grid(dw_conn, layout=layout)

The gridmap of the historical average looks quite similar to Scorsese's, with most faces framed in middle-top or middle-center. So what's going on? We can also compare Scorsese's framing compared to the historical average by taking a difference of the two gripmaps. 

In [ ]:
compare_director_to_sample_grid("Martin Scorsese", dw_conn, layout=layout)

As we can see, Scorsese places fewer faces in the middle-top and middle-center cells than the historical average. In conceptual terms, Scorsese's framing is much more "even" or "dispersed" than the historical average--in fact, more than the sample as a whole. Differencing the historical gridmap from Scorsese's tells us *how* Scorsese's gini score is so low, but it doesn't explain *why*. There is another metric, however, that explains why the gini score is so low.

In [ ]:
x = "avg_faces_per_frame_mean"
y = "z_composition_gini_mean"
fig = px.scatter(g, 
                 x=x,
                 y=y,
                 hover_name="person_name_max",
                 color_discrete_sequence=["#79b8b8"],
                 trendline="ols")
fig.update_layout(layout)
fig.update_layout(
    xaxis=dict(title=x), 
    yaxis=dict(title=y), 
    title=dict(
        text=f"{x} vs. {y}",
        y=0.95
    )
)
fig.update_traces(marker=dict(size=8, line=dict(width=1, color="white")))
fig.update_traces(selector=dict(mode="lines"), line=dict(dash="dash", color="#d81275", width=5), name="Trend")
temp = g[g['person_name_max'] == "Martin Scorsese"]
fig.add_trace(go.Scatter(x=temp[x], 
                         y=temp[y], 
                         marker=dict(
                             color="red", 
                             size=16, 
                             line=dict(
                                 color="white",
                                 width=2
                             )
                             )))
fig.show()

If we look at the average number of faces in a frame, we see that Scorsese likes to place many more faces in the frame than the historical average. This forms an interesting relationship with the average face size and the face size variance. Those two metrics are right around the mean, which means that Scorsese is not adding more faces at the expense of size, as in he doesn't move the camera farther away when adding more faces but instead "crowds" the frame.

In [ ]:
x = "z_avg_face_size"
y = "avg_faces_per_frame"
temp = df[df['person_name'] == "Martin Scorsese"]
fig = px.scatter(temp, 
                 x=x,
                 y=y,
                 hover_name="person_name",
                 hover_data="title",
                 color_discrete_sequence=["#79b8b8"],
                 trendline="ols")
fig.update_layout(layout)
fig.update_layout(
    xaxis=dict(title=x), 
    yaxis=dict(title=y), 
    title=dict(
        text=f"{x} vs. {y}",
        y=0.95
    )
)
fig.update_traces(marker=dict(size=8, line=dict(width=1, color="white")))
fig.update_traces(selector=dict(mode="lines"), line=dict(dash="dash", color="#d81275", width=5), name="Trend")
fig.add_trace(go.Scatter(
    x=temp[x], 
    y=[temp[y].mean() for _ in temp.values], 
    mode='lines', 
    marker=dict(color="purple"), 
    line=dict(width=4)))
fig.show()

Scorsese has roughly and equal number of films above the mean as below. His average is being pulled up by a few films, particularly *The Aviator*, *Gangs of New York*, and *Taxi Driver*. It would be interesting to see if these averages change significantly with more films added.

#### Ingmar Bergman -- Master of the Close-Up

With regard to face size and variance of face size, Ingmar Bergman is king. He is an extreme outlier, packing the frame with large faces.

In [ ]:
x = "z_size_mean"
y = "z_v_size_mean"
fig = px.scatter(g, 
                 x=x,
                 y=y,
                 hover_name="name",
                 color_discrete_sequence=["#79b8b8"],
                 trendline="ols")
fig.update_layout(layout)
fig.update_layout(
    xaxis=dict(title=x), 
    yaxis=dict(title=y), 
    title=dict(
        text=f"{x} vs. {y}",
        y=0.95
    )
)
fig.update_traces(marker=dict(size=8, line=dict(width=1, color="white")))
fig.update_traces(selector=dict(mode="lines"), line=dict(dash="dash", color="#d81275", width=5), name="Trend")
temp = g[g['name'] == "Ingmar Bergman"]
fig.add_trace(go.Scatter(x=temp[x], 
                         y=temp[y], 
                         marker=dict(
                             color="red", 
                             size=16, 
                             line=dict(
                                 color="white",
                                 width=2
                             )
                             )))
fig.show()

Bergman is #1 in both average size and average variance. Bergman has a consistently high score for size and variance, but he also has a greater range between films than most other directors.

In [ ]:
x = "z_avg_face_size_range"
y = "z_variance_face_size_range"
fig = px.scatter(g, 
                 x=x,
                 y=y,
                 hover_name="person_name_max",
                 color_discrete_sequence=["#79b8b8"],
                 trendline="ols")
fig.update_layout(layout)
fig.update_layout(
    xaxis=dict(title=x), 
    yaxis=dict(title=y), 
    title=dict(
        text=f"{x} vs. {y}",
        y=0.95
    )
)
fig.update_traces(marker=dict(size=8, line=dict(width=1, color="white")))
fig.update_traces(selector=dict(mode="lines"), line=dict(dash="dash", color="#d81275", width=5), name="Trend")
temp = g[g['person_name_max'] == "Ingmar Bergman"]
fig.add_trace(go.Scatter(x=temp[x], 
                         y=temp[y], 
                         marker=dict(
                             color="red", 
                             size=16, 
                             line=dict(
                                 color="white",
                                 width=2
                             )
                             )))
fig.show()

As we can see, Bergman is in the top 5 for range. We can see below to see the average across Bergman's films contained in the sample

In [ ]:
temp = df[df['person_name'] == "Ingmar Bergman"].sort_values(by='z_avg_face_size', ascending=False)
fig = px.bar(temp, x='title', y=['z_avg_face_size', 'z_variance_face_size'], barmode="group")
fig.update_layout(layout)
fig.show()

Despite this wide range, none of his films contained in the sample are below average for either face size or variance. *Autumn Sonata* is his most extreme film, in the top 10 for both size and variance.

#### Orson Welles

In [ ]:
name = "Orson Welles"

##### Face Size vs. Face Size Variance

In [ ]:
x = "z_size_g_mean"
y = "z_v_size_g_mean"
plot_director_against_sample(g, x, y, name)

##### Average Distance from Center vs. Variance Distance from Center

Orson Welles positions his faces farther from the center of the frame than any other director. 

In [ ]:
x = "z_dist_g_mean"
y = "z_v_dist_g_mean"
plot_director_against_sample(g, x, y, name)

##### Average Distance from Center Global vs. Variance Distance from Center Global

But if we normalize by year, a different story emerges. Not only does Welles de-center his images more than anyone else, he is on an island all by himself. Welles is incredibly unique in classical Hollywood. 

In [ ]:
x = "z_dist_mean"
y = "z_v_dist_mean"
plot_director_against_sample(g, x, y, name)

In [ ]:
x = "z_gini_g_mean"
y = "z_disp_g_mean"
plot_director_against_sample(g, x, y, "Orson Welles")

#### Alfred Hitchcock

In [ ]:
name = "Alfred Hitchcock"

If we compare Hitchcock to the global averages, he is 

In [ ]:
x = "z_size_g_mean"
y = "z_v_size_g_mean"
plot_director_against_sample(g, x, y, name)

Compared to directors of his era, Hitchcock more frequently uses close shots. 

In [ ]:
x = "z_size_mean"
y = "z_v_size_mean"
plot_director_against_sample(g, x, y, name)

In [ ]:
g['size_percentile'] = g['z_size_mean'].rank(pct=True) * 100
temp = g[g['name'] == name]
print(temp['size_percentile'].values[0])

g['size_v_percentile'] = g['z_v_size_mean'].rank(pct=True) * 100
temp = g[g['name'] == name]
print(temp['size_v_percentile'].values[0])

89.83739837398373
88.6178861788618


In [ ]:
x = "z_size_std"
y = "z_v_size_std"
plot_director_against_sample(g, x, y, name)

In [ ]:
g['size_percentile'] = g['z_size_std'].rank(pct=True) * 100
temp = g[g['name'] == name]
print(temp['size_percentile'].values[0])

g['size_v_percentile'] = g['z_v_size_mean'].rank(pct=True) * 100
temp = g[g['name'] == name]
print(temp['size_v_percentile'].values[0])

In [ ]:
x = "z_size_g_std"
y = "z_v_size_g_std"
plot_director_against_sample(g, x, y, name)

In [ ]:
x = "z_dist_g_mean"
y = "z_v_dist_g_mean"
plot_director_against_sample(g, x, y, name)

In [ ]:
x = "z_dist_mean"
y = "z_v_dist_mean"
plot_director_against_sample(g, x, y, name)

In [ ]:
x = "z_dist_std"
y = "z_v_dist_std"
plot_director_against_sample(g, x, y, name)

NameError: name 'plot_director_against_sample' is not defined

#### Christopher Nolan

In [ ]:
name = "Christopher Nolan"

##### Size vs. Size Variance

In [ ]:
x = "z_size_g_mean"
y = "z_v_size_g_mean"
plot_director_against_sample(g, x, y, name)

#### Ozu

In [ ]:
name = "Yasujirō Ozu"

##### Size vs. Size Variance

In [ ]:
x = "z_size_g_mean"
y = "z_v_size_g_mean"
plot_directors_against_sample(g, x, y, [name, "Christopher Nolan"])

##### Size vs. Size Variance Normalized by Year

In [ ]:
x = "z_size_mean"
y = "z_v_size_mean"
plot_directors_against_sample(g, x, y, [name, "Christopher Nolan"])

##### Distance Variance vs. Gini 

In [ ]:
x = "z_v_dist_g_mean"
y = "z_gini_g_mean"
plot_directors_against_sample(g, x, y, [name, "Christopher Nolan"])

NameError: name 'plot_directors_against_sample' is not defined

##### Vertical Discipline vs. Gini

In [ ]:
x = "z_vert_g_mean"
y = "z_gini_mean"
plot_directors_against_sample(g, x, y, [name, "Christopher Nolan"])

##### Faces per Frame vs. Dispersion

In [ ]:
x = "z_f_per_fr_g_mean"
y = "z_disp_g_mean"
plot_directors_against_sample(g, x, y, [name, "Christopher Nolan"])

##### Faces per Frame vs. Distance Variance

In [ ]:
x = "z_f_per_fr_g_mean"
y = "z_v_dist_g_mean"
plot_directors_against_sample(g, x, y, [name, "Christopher Nolan"])

In [ ]:
x = "z_gini_g_mean"
y = "z_h_spread_mean"
plot_directors_against_sample(g, x, y, [name, "Christopher Nolan"])

In [ ]:
x = "z_disp_g_mean"
y = "z_h_spread_g_mean"
plot_directors_against_sample(g, x, y, [name, "Christopher Nolan"])

In [ ]:
features={"x": "z_f_per_fr_g_mean", "y": "z_v_dist_g_mean", "z": "z_gini_g_mean"}
plot_directors_3d(g, [name, "Christopher Nolan"], features=features)

#### Kenji Mizoguchi

In [ ]:
name = "Kenji Mizoguchi"

##### Size vs. Size Variance

In [ ]:
x = "z_size_g_mean"
y = "z_v_size_g_mean"
plot_directors_against_sample(g, x, y, [name])

##### Size vs. Distance Variance

In [ ]:
x = "z_size_g_mean"
y = "z_v_dist_g_mean"
plot_directors_against_sample(g, x, y, [name])

##### Faces per Frame vs. Gini

In [ ]:
x = "z_f_per_fr_g_mean"
y = "z_gini_g_mean"
plot_directors_against_sample(g, x, y, [name])

##### Faces per Frame vs. Dispersion

In [ ]:
x = "z_f_per_fr_g_mean"
y = "z_disp_g_mean"
plot_directors_against_sample(g, x, y, [name])